In [1]:
import nest_asyncio  # 충돌 막는다! vs code랑 jupyter가 중복 실행 해서 느려짐 --> 그걸 방지
import os # 운영체제(ex.windows)에 접근할 수 있도록 도와주는 라이브러리
from dotenv import load_dotenv  # .env파일에 있는 보안 내용을 쉽게 가져오게 해주는 라이브러리

nest_asyncio.apply() # 충돌 막는 함수 적용

load_dotenv()

API_KEY = os.getenv('OPENAPI_API_KEY')  # 운영체제에 등록된 환경변수를 읽어 새로운 변수에 담는다.


In [2]:
# ✅ 공통 설정 - 모든 실습에서 사용
import requests # 웹페이지에 있는 내용을 쉽게 가져오도록 해주는 라이브러리
import json # json 파일을 다루기 쉽게 도와주는 라이브러리
import pandas as pd
from datetime import datetime  # 시간과 날짜에 관련된 설정이 있는 라이브러리


def pretty(data):
    """JSON 보기 좋게 출력"""
    print(json.dumps(data, ensure_ascii=False, indent=2))

def check_response(response):
    """응답 상태 확인"""
    if response.status_code == 200:
        print("✅ 성공! (200)")
    elif response.status_code == 401:
        print("🔴 API 키 오류 (401) - 키를 확인하세요")
    elif response.status_code == 400:
        print("🔴 파라미터 오류 (400) - 필수값을 확인하세요")
    else:
        print(f"🔴 오류: {response.status_code}")
    return response.status_code == 200

print("공통 설정 완료! ✅")

공통 설정 완료! ✅


---
# 🍽️ 대구 푸드 정보

**포털 검색:** `대구맛집`  


> 이 API는 각 지자체에서 제공하므로 **지역별로 별도 신청** 필요!  
> 대구시: `대구 일반음식점` 으로 검색
```

In [3]:
# https://www.data.go.kr/data/15057236/openapi.do

daegu_food = 'https://www.daegufood.go.kr/kor/api/tasty.html?mode=json&addr=%EC%A4%91%EA%B5%AC'

daegu_food

'https://www.daegufood.go.kr/kor/api/tasty.html?mode=json&addr=%EC%A4%91%EA%B5%AC'

In [4]:
url = 'https://www.daegufood.go.kr/kor/api/tasty.html' # 대구 맛집 주소

params = {
    'mode':'json',
    'addr':'중구'
}   # 결과 전달 형태 json, 검색어 중구


In [5]:
daegu_food_j = requests.get(url, params=params)

check_response(daegu_food_j)

✅ 성공! (200)


True

In [6]:
# 🍽️ 대구 맛집 데이터 → DataFrame 변환
food_data = daegu_food_j.json()  # JSON으로 파싱

# 응답 구조 확인: data > list 안에 음식점 목록이 있음
food_items = food_data['data']   # 음식점 목록 추출

# DataFrame 생성
df_food = pd.DataFrame(food_items)

print(f'총 {len(df_food)}개 음식점 데이터 로드 완료!')
print(f'컬럼 목록: {df_food.columns.tolist()}')


총 178개 음식점 데이터 로드 완료!
컬럼 목록: ['cnt', 'OPENDATA_ID', 'GNG_CS', 'FD_CS', 'BZ_NM', 'TLNO', 'MBZ_HR', 'SEAT_CNT', 'PKPL', 'HP', 'PSB_FRN', 'BKN_YN', 'INFN_FCL', 'BRFT_YN', 'DSSRT_YN', 'MNU', 'SMPL_DESC', 'SBW', 'BUS']


In [7]:
# 📋 컬럼명 한글로 변환 (가독성 향상)
col_rename = {
    'cnt':        '순번',
    'OPENDATA_ID':'ID',
    'GNG_CS':     '주소',
    'FD_CS':      '음식분류',
    'BZ_NM':      '업소명',
    'TLNO':       '전화번호',
    'MBZ_HR':     '영업시간',
    'SEAT_CNT':   '좌석수',
    'PKPL':       '주차',
    'HP':         '홈페이지',
    'PSB_FRN':    '가능언어',
    'BKN_YN':     '예약가능',
    'INFN_FCL':   '영유아시설',
    'BRFT_YN':    '조식여부',
    'DSSRT_YN':   '디저트여부',
    'MNU':        '메뉴',
    'SMPL_DESC':  '업소소개',
    'SBW':        '지하철정보',
    'BUS':        '버스정보',
}

df_food_kr = df_food.rename(columns=col_rename)

# 핵심 컬럼만 보기
df_view = df_food_kr[['순번', '업소명', '음식분류', '주소', '전화번호', '영업시간', '좌석수', '예약가능']]
df_view


,순번,업소명,음식분류,주소,전화번호,영업시간,좌석수,예약가능
0,1,공주당베이커리,디저트/베이커리,대구광역시 중구 공평동 11-3,053-427-7256,08:00 ~ 20:00,없음,가능
1,2,톤톤 돈카츠,일식,대구광역시 중구 대봉동 4-1,053-252-4668,11:00 ~ 20:30 (브레이크타임 14:40 ~ 17:30),20석,가능
2,3,육즙,한식,대구광역시 중구 대봉동 9-6,053-262-6060,16:00 ~ 01:00 (주말 14:00 ~ 01:00),24석,가능
3,4,음밀한양,중식,대구광역시 중구 대봉동 2-11,0507-1353-2337,16:30 ~ 23:50,40석,가능
4,5,와래이수제꼬치전문점,일식,대구광역시 중구 대봉동 3-16,053-254-1553,16:00 ~ 01:00 (주말 12:00 ~ 01:00),10석,가능
...,...,...,...,...,...,...,...,...
173,174,너구리,한식,대구광역시 중구 향촌동 74-7,053-427-9292,10:00 ~ 22:00,60석,가능
174,175,국일생갈비,한식,대구광역시 중구 동산동 106-1,053-254-5115,11:30 ~ 21:30,160석(룸9),가능
175,176,국일따로국밥,한식,대구광역시 중구 전동 7-1,053-253-7623,24시간,150석(룸1),가능
176,177,교동따로식당,한식,대구광역시 중구 포정동 52-2,053-254-8923,24시간,60석,가능


In [8]:
# 📊 음식 분류별 업소 수 집계
print('=== 음식 분류별 업소 수 ===')
print(df_food_kr['음식분류'].value_counts())
print()

# 특정 분류만 필터링 (예: 한식)
df_hansik = df_food_kr[df_food_kr['음식분류'] == '한식']
print(f'\n🥢 한식 업소 ({len(df_hansik)}곳):')
print(df_hansik[['업소명', '주소', '영업시간']].to_string(index=False))


=== 음식 분류별 업소 수 ===
음식분류
한식           112
일식            18
세계요리          11
중식             8
전통차/커피전문점      8
디저트/베이커리       7
특별한 술집         7
양식             7
Name: count, dtype: int64


🥢 한식 업소 (112곳):
         업소명                       주소                                  영업시간
          육즙         대구광역시 중구 대봉동 9-6      16:00 ~ 01:00 (주말 14:00 ~ 01:00)
        초가식당       대구광역시 중구 동성로1가 2-1         11:30 ~ 21:00 (15:30 ~ 17:00)
         파슬리        대구광역시 중구 대봉동 7-38     11:30 ~ 21:00 (평일만 15:00 ~ 17:00)
    반월당 부자식당       대구광역시 중구 남산동 938-8 11:00 ~ 22:50 (브레이크 타임 15:30 ~ 17:00)
       장모님국밥     대구광역시 중구 삼덕동2가 149-6                         09:00 ~ 21:00
         춘천옥         대구광역시 중구 동인동4가 4                         09:00 ~ 19:30
       청해회수산       대구광역시 중구 대봉동 17-10                         16:00 ~ 00:00
         남산면       대구광역시 중구 남산동 938-9                         11:00 ~ 21:00
        동아식당        대구광역시 중구 공평동 10-2                         11:30 ~ 21:00
        모두국밥     대구광역시 중구 동인동2가 61-

In [9]:
# 💾 CSV로 저장 (엑셀에서도 열 수 있음)
df_food_kr.to_csv('대구_중구_맛집.csv', index=False, encoding='utf-8-sig')  # utf-8-sig: 엑셀 한글 깨짐 방지
print('✅ 대구_중구_맛집.csv 저장 완료!')
print(f'   → {len(df_food_kr)}행 × {len(df_food_kr.columns)}열')

# 저장된 파일 확인
df_check = pd.read_csv('대구_중구_맛집.csv')
df_check.head(3)


✅ 대구_중구_맛집.csv 저장 완료!
   → 178행 × 19열


,순번,ID,주소,음식분류,업소명,전화번호,영업시간,좌석수,주차,홈페이지,가능언어,예약가능,영유아시설,조식여부,디저트여부,메뉴,업소소개,지하철정보,버스정보
0,1,1890,대구광역시 중구 공평동 11-3,디저트/베이커리,공주당베이커리,053-427-7256,08:00 ~ 20:00,없음,없음,없음,영어,가능,불가능,불가능,불가능,"쌀빵 3,000원 <br />팥빵(4개) 3,000원<br />시몬카스테라(6개) ...",공주당 베이커리 는 동성로 한가운데에서 복고적인 분위기와 함께 정감 있는 맛을 전하...,지하철 1호선 중앙로역 3번 출구에서 도보로 약 556m 거리,버스 정류장은 대구시티센터 정류장이 가장 가깝습니다.
1,2,1874,대구광역시 중구 대봉동 4-1,일식,톤톤 돈카츠,053-252-4668,11:00 ~ 20:30 (브레이크타임 14:40 ~ 17:30),20석,김광석길공영주차장이용(유료),없음,영어,가능,불가능,불가능,불가능,"히레카츠(안심) 12,000원<br />로스카츠(등심) 12,000원<br />치즈...",‘톤톤 돈가츠’는 작지만 정성 가득한 수제 돈카츠 전문점입니다.,지하철 2호선 경대병원역 3번 출구에서 도보로 약 483m 거리.,버스 정류장은 방천시장(김광석길)앞 정류장이 가장 가깝습니다.
2,3,1873,대구광역시 중구 대봉동 9-6,한식,육즙,053-262-6060,16:00 ~ 01:00 (주말 14:00 ~ 01:00),24석,김광석길공영주차장이용,없음,영어,가능,불가능,불가능,불가능,"본삼겹(150g) 18,000원 <br />삼겹살(120g) 13,000원 <br ...",육즙 은 숙성 삼겹살과 다양한 부위의 숯불구이를 전문으로 하는 고깃집입니다.,지하철 2호선 경대병원역 3번 출구에서 도보로 약 498m 거리.,버스 정류장은 방천시장(김광석길)앞 정류장이 가장 가깝습니다.
